## Growth Modeling
This notebook models playlist follower growth / outputs model coefficient tables using the metadata features we collected.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, root_mean_squared_error

### Load features and follower count (day by day)

In [ ]:
playlist_features_df = pd.read_csv("data/processed/playlist_features.csv")
playlist_features_df.drop(columns=["playlist_id"], errors="ignore").head()

In [ ]:
follower_files = sorted(glob.glob("data/raw/playlist_followers_*.csv"))
if len(follower_files)<2:
    raise ValueError("at least 2 follower files are needed to calculate growth.")

followers_df = pd.concat([pd.read_csv(f) for f in follower_files], ignore_index=True)
followers_df["snapshot_date"] = pd.to_datetime(followers_df["snapshot_date"])
followers_df = followers_df.sort_values(["playlist_id", "snapshot_date"])

# followers_df.drop(columns=["playlist_id"], errors="ignore").head()

### Growth targets

In [ ]:
# determine change since prev snapshot

followers_df["followers_prev"] = (followers_df.groupby("playlist_id")["followers"].shift(1))
followers_df["delta_followers"] = (followers_df["followers"]-followers_df["followers_prev"])

followers_df["log_followers"] = np.log1p(followers_df["followers"])
followers_df["log_followers_prev"] = np.log1p(followers_df["followers_prev"])
followers_df["delta_log_followers"] = (followers_df["log_followers"]-followers_df["log_followers_prev"])

growth_df = followers_df.dropna(subset=["delta_followers", "delta_log_followers"])

model_df = growth_df.merge(playlist_features_df,on="playlist_id",how="left")

In [ ]:
target = "delta_log_followers"
# target = "delta_followers"        # absolute growth

excl_cols = {
    "playlist_id", 
    "playlist_name",
    "followers",
    "followers_prev",
    "delta_followers",
    "delta_log_followers",
}

feature_cols = [c for c in model_df.columns if c not in excl_cols]


X = model_df[feature_cols]
y = model_df[target]

numeric_feature_cols = X.select_dtypes(include=["number"]).columns.tolist()
X = X[numeric_feature_cols]


### Train / test split

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)

### Linear regression

In [ ]:
n_samples = len(X)
print("# samples:", n_samples)

lin_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

if n_samples < 5:
    lin_pipe.fit(X, y)
    preds = lin_pipe.predict(X)

    results = {
        "r2": r2_score(y, preds) if n_samples > 1 else None,
        "rmse": root_mean_squared_error(y, preds) if n_samples > 1 else None
    }

    results_df = pd.DataFrame([results])

else:
    tscv = TimeSeriesSplit(n_splits=min(3, n_samples - 1))

    lin_scores = []
    for tr, te in tscv.split(X):
        lin_pipe.fit(X.iloc[tr], y.iloc[tr])
        preds = lin_pipe.predict(X.iloc[te])

        fold_results = {
            "rmse": root_mean_squared_error(y.iloc[te], preds)
        }

        if len(te) >= 2:
            fold_results["r2"] = r2_score(y.iloc[te], preds)
        else:
            fold_results["r2"] = None

        lin_scores.append(fold_results)

    results_df = pd.DataFrame(lin_scores)

results_df

### Ridge regression ($l_2$ regularization)

In [ ]:
ridge_pipe = Pipeline([("scaler", StandardScaler()),("model", Ridge(alpha=1.0))])

ridge_scores = []
for tr, te in tscv.split(X):
    ridge_pipe.fit(X.iloc[tr], y.iloc[tr])
    preds = ridge_pipe.predict(X.iloc[te])

    ridge_scores.append({
        "r2": r2_score(y.iloc[te], preds),
        "rmse": root_mean_squared_error(y.iloc[te], preds)
    })

pd.DataFrame(ridge_scores)

Fit model on all data

In [ ]:
ridge_pipe.fit(X, y)

coef_df = pd.DataFrame({"feature": X.columns,"coefficient": ridge_pipe.named_steps["model"].coef_}).sort_values("coefficient", ascending=False)
coef_df.head(15)


### Save outputs

In [ ]:
out_dir = "data/processed"
os.makedirs(out_dir, exist_ok=True)

growth_path = os.path.join(out_dir, "growth_targets.csv")
coef_path = os.path.join(out_dir, "growth_model_coefficients.csv")

growth_df.to_csv(growth_path, index=False)
coef_df.to_csv(coef_path, index=False)
growth_path, coef_path